# GVH Diagonal Cubic — Action-candidate two-cube shear-overlap gate (FAST)

**Mission.** Chercher sans réglage une structure de couplage à partir de l'action *réelle* `S_cand` archivée dans `.3.3.28`, puis vérifier algébriquement et numériquement cette structure. Utiliser le benchmark Terre–Lune comme contrôle externe non ajustable, **sans prétendre calculer une correction GVH de la gravitation Terre–Lune**.

**Architecture obligatoire.** 1 — GVH ; 2 — physique établie ; 3 — test numérique minimal ; 4 — verdict scientifique.

**Périmètre strict :** calcul du **terme croisé de la densité d'action** lorsque l'on *suppose* une décomposition additive du cisaillement `sigma = sigma_A + sigma_B` à trièdre local partagé. La décomposition additive est un *ansatz de test*, non une solution des équations d'Einstein-æther/GVH. Les profils gaussiens sont des données artificielles sans dimension pour vérifier l'algèbre, **pas** des profils Terre/Lune mesurés.

**Verrous non franchis :** terme de matière/source et conditions au bord ; solutions de champs à deux sources ; couplages `c_i`, `zeta_C` dérivés ; relation `G_0`/`G_N` ; comparaison SI non circulaire ; stabilité globale ; prédiction d'un changement d'orbite ou de fréquence GVH.

In [1]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
import sympy as sp
ROOT = Path('/mnt/data') if Path('/mnt/data').exists() else (Path('/content') if Path('/content').exists() else Path.cwd())
ROOT.mkdir(exist_ok=True,parents=True)
REF28 = ROOT/'gvh_source/GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb'
REF281 = ROOT/'gvh_source/GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.1_Cubic_Vector_Closure_Metric_Helix_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb'
BENCH = ROOT/'GVH_Hierarchical_Cube_Quadrupole_Frequency_Gate_FAST_results.json'
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
# These hashes were computed on the Library-derived reference copies during notebook creation.
SRC_28_SHA = 'c8015f1d771caa5f65c8ad5d64505733ebfa96a81f6153a8b1d8c2233cfbeefa'
SRC_281_SHA = '0018b54c8f07984e3ebf35ea6989e6772bebb815492b66b672bed83241f8d5b4'
SRC_BENCH_SHA = '08a87715392bdcb4ff44c4de0e01620c21fe1b1322f5bfb0df97102c4a79ce08'
for path, expected in [(REF28,SRC_28_SHA),(REF281,SRC_281_SHA),(BENCH,SRC_BENCH_SHA)]:
    if path.exists():
        assert sha(path)==expected, 'Reference changed: '+str(path)
        print('Local reference verified:', path.name)
    else:
        print('Local reference not mounted; using documented frozen audit excerpt / benchmark snapshot:',path.name)
print('ARCHIVED_ACTION_TERMS: L_C=(zeta_C/2)*sum_a (e_a^mu e_a^nu sigma_mu_nu)^2')
print('ARCHIVED_RIGID_HELIX_GATE: sigma=0 => cubic contribution vanishes')
print('PROVENANCE_SHA256:', SRC_28_SHA,SRC_281_SHA,SRC_BENCH_SHA)


Local reference verified: GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28_Full_GVH_EOM_Derivation_and_Variational_Closure_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb
Local reference verified: GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.28.1_Cubic_Vector_Closure_Metric_Helix_Gate_Audit_FAST_EXECUTED_REFERENCE.ipynb
Local reference verified: GVH_Hierarchical_Cube_Quadrupole_Frequency_Gate_FAST_results.json
ARCHIVED_ACTION_TERMS: L_C=(zeta_C/2)*sum_a (e_a^mu e_a^nu sigma_mu_nu)^2
ARCHIVED_RIGID_HELIX_GATE: sigma=0 => cubic contribution vanishes
PROVENANCE_SHA256: c8015f1d771caa5f65c8ad5d64505733ebfa96a81f6153a8b1d8c2233cfbeefa 0018b54c8f07984e3ebf35ea6989e6772bebb815492b66b672bed83241f8d5b4 08a87715392bdcb4ff44c4de0e01620c21fe1b1322f5bfb0df97102c4a79ce08


# Niveau 1 — GVH : dériver la structure croisée du terme cubique

L'action archivée `.3.3.28` inclut :

\[
S_{\rm cand}=\frac{1}{16\pi G_0}\int d^4x\sqrt{-g}\,[R+\mathcal L_u+\cdots+\tfrac{\zeta_C}{2}\mathcal O_C],\qquad
\mathcal O_C=\sum_{a=1}^{3}s_a^2,\quad s_a=e_a^\mu e_a^\nu\sigma_{\mu\nu}.
\]

L'opérateur dépend de la projection d'un **champ local de cisaillement** et non directement de « deux cubes » étiquetés. Introduisons uniquement pour ce test une décomposition locale **postulée** `sigma = sigma_A + sigma_B` sur le même fond `g,u,e_a`, donc `s_a = s_a^(A)+s_a^(B)` :

\[
\boxed{\Delta\mathcal L_{C,AB}\equiv\mathcal L_C[A+B]-\mathcal L_C[A]-\mathcal L_C[B]
=\zeta_C\sum_a s_a^{(A)}s_a^{(B)}.}
\]

Cette identité est dérivée de l'opérateur de l'action, **conditionnellement** à l'ansatz additif ; elle ne constitue ni un terme fondamental indépendant `S_AB`, ni une solution avec sources, ni une force mesurable.

In [2]:
zeta = sp.symbols('zeta_C', real=True)
a1,a2,a3,b1,b2,b3 = sp.symbols('a1 a2 a3 b1 b2 b3',real=True)
sa = sp.Matrix([a1,a2,a3]); sb = sp.Matrix([b1,b2,b3])
LC = lambda s: zeta*sum(s[i]**2 for i in range(3))/2
cross = sp.expand(LC(sa+sb)-LC(sa)-LC(sb))
expected = zeta*sum(sa[i]*sb[i] for i in range(3))
CROSS_IDENTITY_PASS = sp.simplify(cross-expected)==0
assert CROSS_IDENTITY_PASS
# Two spatial, symmetric, trace-free shear test matrices that are diagonal in the same local triad.
A,B,p,q = sp.symbols('A B p q',real=True)
diagA=sp.Matrix([A*p,-A*p,0]); diagB=sp.Matrix([B*q,-B*q,0])
traceA=sp.simplify(sum(diagA)); traceB=sp.simplify(sum(diagB))
parallel_cross=sp.factor(cross.subs(dict(zip([a1,a2,a3],diagA))).subs(dict(zip([b1,b2,b3],diagB))))
# Orthogonal in the Frobenius / triad-diagonal projection product.
orthoB=sp.Matrix([B*q,B*q,-2*B*q])
orthogonal_cross=sp.simplify(cross.subs(dict(zip([a1,a2,a3],diagA))).subs(dict(zip([b1,b2,b3],orthoB))))
assert traceA==traceB==0 and parallel_cross==2*zeta*A*B*p*q and orthogonal_cross==0
assert sum(orthoB)==0
print('CROSS_IDENTITY_PASS=',CROSS_IDENTITY_PASS)
print('STF_PARALLEL_CROSS_DENSITY=',parallel_cross)
print('ORTHOGONAL_STF_CROSS=',orthogonal_cross)
print('ORIENTATION_CONSTRAINT_TEST: both matrices diagonal, offdiagonal M_AB=0')

CROSS_IDENTITY_PASS= True
STF_PARALLEL_CROSS_DENSITY= 2*A*B*p*q*zeta_C
ORTHOGONAL_STF_CROSS= 0
ORIENTATION_CONSTRAINT_TEST: both matrices diagonal, offdiagonal M_AB=0


## Attention à l'équation du mouvement réelle

L'audit `.3.3.28.1` établit que, **sur une hélice rigide sans cisaillement**, `sigma = 0`, donc `O_C = 0` et la contribution cubique à la première variation s'annule. La simulation Terre–Lune de quadrupôle est un résultat de la **gravitation newtonienne**, pas un champ de cisaillement `sigma` obtenu en résolvant `S_cand`.

Les matrices ci-dessus respectent algébriquement la symétrie et la trace nulle, mais **n'ont pas été reconstruites à partir d'un champ quadrivecteur `u` satisfaisant les EOM couplées avec la métrique**. Il est donc interdit d'appeler cette densité d'action une « force GVH Terre–Lune ».

In [3]:
zero_cross=sp.simplify(cross.subs({a1:0,a2:0,a3:0}))
rigid_shear_zero=zero_cross==0
assert rigid_shear_zero
print('RIGID_SHEAR_FREE_CROSS_IS_ZERO=',rigid_shear_zero)
# No term for matter sources appears in the archived S_cand formula; separate S_m or sourcing needed.
SOURCE_MATTER_ACTION_MATERIALIZED_IN_S_CAND=False
TWO_BODY_FIELD_SOLUTION_DERIVED=False
FULL_METRIC_EOM_TWO_BODY_SOLVED=False
CUBIC_EOM_THEORY_TO_EARTH_MOON_READY=False
assert not all([SOURCE_MATTER_ACTION_MATERIALIZED_IN_S_CAND,TWO_BODY_FIELD_SOLUTION_DERIVED,FULL_METRIC_EOM_TWO_BODY_SOLVED])

RIGID_SHEAR_FREE_CROSS_IS_ZERO= True


# Niveau 2 — Physique établie : un contrôle séparé, sans réglage

Nous relisons le benchmark numérique newtonien Terre–Lune déjà exécuté. Il utilise les GM de JPL et une distance de référence NASA et prédit l'harmonique `2 f_orbit` du quadrupôle pour une orbite circulaire. Ce résultat relève de la **physique établie** et ne fixe ni `zeta_C` ni les `c_i` de l'action candidate.

Même si les deux mécanismes peuvent générer des variations périodiques, l'égalité de fréquences seule **n'identifie pas** le mécanisme. Le secteur de matière et des solutions des EOM restent nécessaires pour comparer les amplitudes en unités SI.

In [4]:
bench=json.loads('''{"notebook":"GVH_Hierarchical_Cube_Quadrupole_Frequency_Gate_FAST","canonical_status":"COMPLEMENTARY_NOT_CANONICAL","upstream_file_sha256":"4b7230bb557b951abe6547a62dc428aeccc78e6d84cddb2506810ca94c78f5b5","four_levels":{"level1_GVH":"HIERARCHY_PROPOSED__INTERCUBE_EOM_NOT_DERIVED_HERE","level2_established_physics":"NEWTONIAN_MULTIPOLE_QUADRUPOLE_BENCHMARK","level3_numeric":"FAR_FIELD_APPROXIMATION_AND_2F_SPECTRAL_WITNESS","level4_scientific_verdict":"NEWTONIAN_BENCHMARK_PASS__GVH_DISCRIMINANT_BLOCKED"},"constants":{"GM_E_m3s2":398600435507000.0,"GM_M_m3s2":4902800118000.0,"distance_m":384400000.0,"T_moon_reference_days":27.3217},"results":{"checks":{"symbolic_harmonic":true,"barycenter_identity":true,"quadrupole_improves_all_tested_distances":true,"dominant_harmonic_index_2":true,"harmonic_amplitude_QA_pass":true,"primary_quadrupole_improvement_pass":true,"distance_holdout_improvement_pass":true},"distance_table":[{"R_m":1000000000.0,"d_over_R":0.3844,"monopole_max_error_pct":1.0267438051241444,"monopole_quad_max_error_pct":0.4946650407920011,"improvement_factor":2.0756344606043715},{"R_m":10000000000.0,"d_over_R":0.03844,"monopole_max_error_pct":0.005600102652027713,"monopole_quad_max_error_pct":0.0002793150086990898,"improvement_factor":20.049415454293708},{"R_m":20000000000.0,"d_over_R":0.01922,"monopole_max_error_pct":0.0013642649137907059,"monopole_quad_max_error_pct":3.406800296904591e-05,"improvement_factor":40.045344455038155},{"R_m":100000000000.0,"d_over_R":0.003844,"monopole_max_error_pct":5.3475226363350575e-05,"monopole_quad_max_error_pct":2.673499281750799e-07,"improvement_factor":200.0196025051152}],"f_internal_Hz":4.236220320870983e-07,"f_quadrupole_predicted_Hz":8.472440641741966e-07,"f_fft_peak_Hz":8.472440641741966e-07,"fft_harmonic_index":2,"quadrupole_amplitude_analytic_ms2":1.6102162726152927e-10,"FFT_k2_amplitude_exact_ms2":1.6118111866968772e-10,"FFT_amp_relative_gap":0.0009904968100924694},"locks":{"GVH_intercube_EOM_derived":false,"GVH_SI_scale_derived":false,"GVH_micro_macro_coupling_derived":false,"new_GVH_physics_validated":false},"pass":true,"next_authorized":"DERIVE_GVH_INTERCUBE_COUPLING_FROM_S_CAND_AND_COMPARE"}''')
if BENCH.exists():
  assert json.loads(BENCH.read_text()) == bench
assert bench['results']['checks']['dominant_harmonic_index_2']
assert not bench['locks']['GVH_intercube_EOM_derived']
summary=pd.DataFrame(bench['results']['distance_table'])
print(summary[['R_m','monopole_max_error_pct','monopole_quad_max_error_pct']].to_string(index=False))
print('f_internal_Hz =',bench['results']['f_internal_Hz'])
print('f_quadrupole_Hz =',bench['results']['f_quadrupole_predicted_Hz'])
print('f_peak_Hz =',bench['results']['f_fft_peak_Hz'])
print('GRAVITATIONAL_BASELINE_READ_ONLY=True')

         R_m  monopole_max_error_pct  monopole_quad_max_error_pct
1.000000e+09                1.026744                 4.946650e-01
1.000000e+10                0.005600                 2.793150e-04
2.000000e+10                0.001364                 3.406800e-05
1.000000e+11                0.000053                 2.673499e-07
f_internal_Hz = 4.236220320870983e-07
f_quadrupole_Hz = 8.472440641741966e-07
f_peak_Hz = 8.472440641741966e-07
GRAVITATIONAL_BASELINE_READ_ONLY=True


# Niveau 3 — Test numérique minimal : superposition, séparation, et fréquences

**Surrogate dimensionless.** Prenons deux profils de cisaillement imposés, de même largeur `w`, séparés de `d` :

\[
p(x)=e^{-(x+d/2)^2/(2w^2)},\quad q(x)=e^{-(x-d/2)^2/(2w^2)}.
\]

Pour les matrices STF co-alignées `diag(Ap,-Ap,0)` et `diag(Bq,-Bq,0)`, le terme croisé intégré sur `x` vaut formellement par unité des autres coordonnées :

\[
\mathcal I(d)=2\zeta_C AB\int p(x)q(x)\,dx
=2\zeta_C AB\sqrt\pi w\,e^{-d^2/(4w^2)}.
\]

Le **rapport normalisé** `I(d)/I(0) = exp[-(d/w)^2/4]` ne dépend ni de `A,B,zeta_C` ni de l'unité de longueur commune : il vérifie uniquement cette construction imposée, pas la loi GVH pour les corps physiques.

In [5]:
x,d,w=sp.symbols('x d w',real=True,positive=True)
p=sp.exp(-(x+d/2)**2/(2*w**2));q=sp.exp(-(x-d/2)**2/(2*w**2))
product_factor=sp.simplify(sp.expand((x+d/2)**2 + (x-d/2)**2))
assert product_factor==2*x*x+d*d/2
integral=sp.sqrt(sp.pi)*w*sp.exp(-d*d/(4*w*w))
ratio=sp.simplify(integral/integral.subs(d,0))
assert ratio==sp.exp(-d*d/(4*w*w))
print('analytic_integral=',integral)
print('analytic_ratio=',ratio)

analytic_integral= sqrt(pi)*w*exp(-d**2/(4*w**2))
analytic_ratio= exp(-d**2/(4*w**2))


In [6]:
# No regression, fit, or optimization; analytic values vs independently quadrature-integrated products.
W=1.0; X=np.linspace(-14,14,200001); test=[]
for D in (0.,1.,2.,4.,6.):
  ga=np.exp(-(X+D/2)**2/(2*W**2))
  gb=np.exp(-(X-D/2)**2/(2*W**2))
  numeric=float(np.trapezoid(ga*gb,X))
  analytic=float(np.sqrt(np.pi)*W*np.exp(-D*D/(4*W**2)))
  row={'d_over_w':D,'overlap_numeric':numeric,'overlap_analytic':analytic,
       'ratio_analytic':np.exp(-D*D/4),'relative_error':abs(numeric-analytic)/analytic}
  test.append(row)
profile_df=pd.DataFrame(test)
NUMERIC_PROFILE_PASS=bool((profile_df.relative_error<1e-10).all())
assert NUMERIC_PROFILE_PASS
print(profile_df.to_string(index=False))
print('NUMERIC_PROFILE_PASS=',NUMERIC_PROFILE_PASS)

 d_over_w  overlap_numeric  overlap_analytic  ratio_analytic  relative_error
      0.0         1.772454          1.772454        1.000000    0.000000e+00
      1.0         1.380388          1.380388        0.778801    0.000000e+00
      2.0         0.652049          0.652049        0.367879    0.000000e+00
      4.0         0.032464          0.032464        0.018316    2.137437e-16
      6.0         0.000219          0.000219        0.000123    1.239155e-16
NUMERIC_PROFILE_PASS= True


## Test séparé de mélange fréquentiel (illustration, non prédiction)

Si les deux cisaillements **imposés** ont des dépendances cosinus `cos(2π f_A t)` et `cos(2π f_B t)`, leur produit comporte une fréquence de différence `|f_A-f_B|` et une fréquence de somme `f_A+f_B`. C'est une identité trigonométrique connue. **Les fréquences ne sont pas déduites des EOM GVH.**

Nous choisissons arbitrairement `f_A=1`, `f_B=1.5` cycles par unité de temps sans dimension. La FFT doit détecter `0.5` et `2.5` cycles par unité de temps. Ce test n'a aucune valeur de calibration SI.

In [7]:
FA=1.0; FB=1.5; DUR=8.; N=8192
T=np.arange(N)*DUR/N
wave=np.cos(2*np.pi*FA*T)*np.cos(2*np.pi*FB*T)
f=np.fft.rfftfreq(N,DUR/N);amp=2*np.abs(np.fft.rfft(wave))/N
order=np.argsort(amp[1:])[::-1][:2]+1
peaks=np.sort(f[order]); expected_peaks=np.array([abs(FA-FB),FA+FB])
FFT_SIDEBAND_PASS=bool(np.allclose(peaks,expected_peaks,atol=1e-12))
assert FFT_SIDEBAND_PASS
print('toy frequency peaks:',peaks,'per dimensionless time unit')
print('FFT_SIDEBAND_PASS=',FFT_SIDEBAND_PASS)
print('these frequencies are prescribed, not GVH-derived')

toy frequency peaks: [0.5 2.5] per dimensionless time unit
FFT_SIDEBAND_PASS= True
these frequencies are prescribed, not GVH-derived


# Niveau 4 — Verdict scientifique / porte de sortie

**Ce qui est dérivé (conditionnel à la décomposition additive) :** `Delta L_C = zeta_C sum_a s_a^A s_a^B`; signe et amplitude dépendent des projections, de `zeta_C` et des solutions de champs.

**Ce qui est testé :** identité symbolique, cisaillements STF co-alignés et orthogonaux, limite sans cisaillement, intégrale analytique de deux profils gaussiens et vérification numérique, mélange fréquentiel sans dimension.

**Ce qui reste BLOQUÉ :** donner une signification physique « Terre–Lune » aux deux profils, calculer une correction au quadrupôle ou au mouvement des corps, prédire une fréquence en Hz, tirer une force `F=-dI/dd` d'une densité d'action non convertie en énergie, déclarer `zeta_C` ou les échelles SI dérivées.

Prochaine porte réellement autorisée : préciser un `S_m[g,psi]`/les sources et conditions au bord ; résoudre une réduction **non shear-free** de l'ensemble EOM `(g,u,e)` ; déterminer les couplages indépendamment ; puis seulement comparer une observable **held-out** avec le benchmark newtonien.

In [8]:
claims = {
 'notebook':'GVH_Action_Candidate_Two_Cube_Shear_Overlap_Gate_FAST',
 'canonical_status':'COMPLEMENTARY_NOT_CANONICAL',
 'four_levels':{
   'level1_GVH':'CONDITIONAL_SHEAR_CROSS_TERM_FROM_S_CAND_ALGEBRA_DERIVED',
   'level2_established_physics':'NEWTONIAN_QUADRUPOLE_REFERENCE_READ_ONLY__TRIG_PRODUCT_STANDARD',
   'level3_numeric':'TOY_OVERLAP_QUADRATURE_AND_FREQUENCY_SIDEBANDS_PASS',
   'level4_scientific_verdict':'ACTION_CROSS_TERM_PASS__PHYSICAL_TWO_BODY_GVH_COUPLING_BLOCKED',
 },
 'provenance':{'action_28_source_sha256':SRC_28_SHA,'helix_28_1_source_sha256':SRC_281_SHA,
   'quadrupole_benchmark_results_sha256':SRC_BENCH_SHA,
   'canonical_executed_28_sha256_quoted_in_28_1':'eff81fd5eea581ea77727f7abf399e19b26db6507f3b3192a559817e69f4c251'},
 'symbolic_checks':{'cross_identity':bool(CROSS_IDENTITY_PASS),'parallel_STF_cross':'2*zeta_C*A*B*p*q',
   'orthogonal_STF_cross_zero':bool(orthogonal_cross==0),'rigid_shear_free_zero':bool(rigid_shear_zero)},
 'numeric':{'profile_rows':profile_df.to_dict(orient='records'),'profile_pass':NUMERIC_PROFILE_PASS,
     'frequency_peaks_toy_dimensionless':peaks.tolist(),'FFT_sidebands_pass':FFT_SIDEBAND_PASS},
 'established_reference':{'f_orbit_Hz':bench['results']['f_internal_Hz'],
     'f_quad_Hz':bench['results']['f_quadrupole_predicted_Hz'],'source':'existing Newtonian JPL+NASA benchmark'},
 'locks':{
   'shear_decomposition_derived_from_field_equations':False,
   'matter_source_sector_materialized':False,
   'coupled_two_body_GVH_EOM_solved':False,
   'GVH_intercube_force_derived':False,
   'GVH_quadrupole_frequency_derived':False,
   'GVH_SI_scale_derived':False,
   'zeta_C_derived':False,
   'new_GVH_physics_validated':False,
 },
 'local_audit_pass':bool(CROSS_IDENTITY_PASS and rigid_shear_zero and NUMERIC_PROFILE_PASS and FFT_SIDEBAND_PASS),
 'next_authorized':'SOURCE_COUPLING_NON_SHEAR_FREE_EOM_SOLUTION_AND_HELD_OUT_GVH_WEAK_FIELD_OBSERVABLE'
}
assert claims['local_audit_pass']
assert not claims['locks']['GVH_intercube_force_derived']
assert not claims['locks']['new_GVH_physics_validated']
p=ROOT/'GVH_Action_Candidate_Two_Cube_Shear_Overlap_Gate_FAST_results.json'
p.write_text(json.dumps(claims,indent=2,ensure_ascii=False,allow_nan=False),encoding='utf-8')
print('LOCAL_AUDIT_PASS=',claims['local_audit_pass'])
print('SCIENTIFIC_VERDICT=',claims['four_levels']['level4_scientific_verdict'])
print('RESULTS_JSON=',p)

LOCAL_AUDIT_PASS= True
SCIENTIFIC_VERDICT= ACTION_CROSS_TERM_PASS__PHYSICAL_TWO_BODY_GVH_COUPLING_BLOCKED
RESULTS_JSON= /mnt/data/GVH_Action_Candidate_Two_Cube_Shear_Overlap_Gate_FAST_results.json
